# GWjax — Two-Phase PE on **real GW150914** with **mlgw_bbh_jax** (SEOBNRv5HM)

This notebook is the **mlgw_bbh_jax** counterpart of
[`gwjax_colab_pe_two_phase.ipynb`](https://github.com/saulo-albuquerque-phys/GWjax/blob/main/examples/gwjax_colab_pe_two_phase.ipynb).
It runs the **GWjax two-phase nested sampler** on **real GW150914**
strain pulled from GWOSC, with the **SEOBNRv5HM** ML surrogate
(`mlgw_bbh_jax`, *model_4*) as the waveform model:

* 7 spherical-harmonic modes (22, 21, 32, 33, 43, 44, 55).
* Aligned spins χ₁, χ₂ ∈ [−0.9, 0.9] (**sampled**, not fixed).
* Mass ratio q = m₁/m₂ ∈ [1, 10].
* Pure-JAX, JIT/vmap/grad-compatible — drops straight into the GWjax sampler.

The sampling dimension is **10**: `m1, m2, chi_1, chi_2, distance, inclination, ra, dec, psi, tc`.
Only `phi_c` (= mlgw's reference orbital phase `phi_0`) is fixed.

The two-phase sampler splits the run into:

1. **Phase 1 (bulk):** a large `num_delete` per iteration, vmap-batched, that
   contracts the prior volume quickly. Runs until the live-evidence
   contribution drops below `phase1_delta_logz_threshold`.
2. **Phase 2 (tail):** classical Skilling NS with `num_delete = 1` from the
   same `NSState`, for an unbiased final evidence integral and tight posterior.

**Runtime → Change runtime type → T4 GPU** before running. The waveform call
is heavier than ripplegw (a small Keras MLP is evaluated for each particle), so
expect a longer JIT compile and a longer steady-state iteration than the
ripplegw two-phase notebook — but with a higher-fidelity waveform.

## 1. Install GWjax (with `[data,mlgw]` extras)

Same private-repo / PAT flow as the other Colab notebooks (Colab Secrets
first, `getpass` fallback). The `mlgw` extra pulls TensorFlow + tf2jax so
the SEOBNRv5HM surrogate can be loaded; the `data` extra pulls gwpy + gwosc
for the real-data fetch.

Generate a PAT at <https://github.com/settings/tokens?type=beta>. A
**fine-grained** token scoped to `saulo-albuquerque-phys/GWjax` with
**Contents: read-only** is enough.

In [ ]:
import os, sys, getpass, subprocess, importlib, importlib.metadata

OWNER, REPO = "saulo-albuquerque-phys", "GWjax"

# ── 0. Pin JAX to the version Colab's CUDA plugin still understands ─────
# Colab's GPU runtimes currently ship jax_cuda12_plugin 0.5.x, which calls
# `register_custom_type_id_handler` — that API was removed in jaxlib 0.10+.
!pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt 2>/dev/null
!pip install -q "jax[cuda12]==0.4.31" "jaxlib==0.4.31"

# ── 1. Auth: GitHub PAT (Colab Secrets first, getpass fallback) ─────────
GH_TOKEN = None
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception:
    pass
if not GH_TOKEN:
    GH_TOKEN = getpass.getpass(f"GitHub PAT (for {OWNER}/{REPO}): ")
GH_TOKEN = (GH_TOKEN or "").strip()
if not GH_TOKEN:
    raise RuntimeError(
        "Empty GitHub PAT — install would fail with 401 from GitHub.\n"
        "Either set the GH_TOKEN Colab secret (key icon in left sidebar, "
        "toggle Notebook access on) or paste a fine-grained PAT scoped to "
        f"{OWNER}/{REPO} with Contents: read-only."
    )

# ── 2. Install gwjax via subprocess so pip output is fully captured ─────
# We inline the token in-process (never goes to the shell as $VAR, which
# avoids the silent-failure mode where shell substitution drops the token).
url = f"gwjax[data,mlgw] @ git+https://{GH_TOKEN}@github.com/{OWNER}/{REPO}.git"
print(f"Installing gwjax[data,mlgw] from {OWNER}/{REPO} (this takes ~3–5 min on Colab)…")
proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-color", url],
    capture_output=True, text=True,
)
del GH_TOKEN, url  # scrub the token before printing

# Print the LAST few KB of pip's stdout + stderr (the build log is huge).
tail = (proc.stdout or "")[-3500:] + "\n--- stderr ---\n" + (proc.stderr or "")[-1500:]
# Defensive: if anything tokenny slipped through (it shouldn't), mask it.
print(tail)

if proc.returncode != 0:
    raise RuntimeError(
        f"pip install gwjax FAILED (exit code {proc.returncode}). "
        "Read the output above for the actual error — likely an invalid/"
        "expired PAT, missing repo permission, or a dep conflict."
    )

# corner is small and noisy errors don't matter as much.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "corner"], check=True)

# ── 3. Verify the install ───────────────────────────────────────────────
try:
    dist = importlib.metadata.distribution("gwjax")
    print(f"\n✓ gwjax {dist.version} installed at {dist.locate_file('gwjax')}")
except importlib.metadata.PackageNotFoundError as e:
    raise RuntimeError(
        "gwjax not in importlib.metadata after a 'successful' pip run — "
        "this should not happen. Re-run the cell."
    ) from e

print("\nIf you just downgraded jax, Runtime → Restart session, then re-run from cell 3.")

### 1b. Pull the vendored `mlgw_bbh_jax` sub-repo (Colab bandaid)

`mlgw_bbh_jax` is shipped as a *vendored* sub-repo at
`<gwjax-install>/mlgw_jax/mlgw_bbh_jax/`. It is currently **not committed to
the main branch** of the GWjax repository that Colab clones from, so the
wheel built on Colab is missing it. Until the sub-repo is committed (or
added as a git submodule), the cell below clones it on the fly into the
install location.

Skip this cell if you're running locally against a working tree that
already has `mlgw_bbh_jax/` populated.

In [ ]:
# ── Vendored-subrepo bandaid for `gwjax[mlgw]` on Colab ────────────────
# The `mlgw_bbh_jax` Python package is a vendored sub-repo that lives at
#     <gwjax-install>/mlgw_jax/mlgw_bbh_jax/
# but it's not currently committed to the main branch of the GWjax repo
# Colab clones from, so the wheel built on Colab is missing it. We:
#
#   1. install `keras-tuner` (training-only dep that the upstream
#      NN_model.py imports eagerly);
#   2. clone the upstream mlgw_bbh_jax into the gwjax install path;
#   3. apply a small inference-only patch on top — the saved .keras model
#      files in TD_models/model_4/ are in Keras 3 format ('batch_shape',
#      'optional'), but Colab's TF 2.13–2.15 stack ships Keras 2. The patch
#      replaces upstream's vanilla `keras.models.load_model` with a manual
#      zip-+-h5py weight loader that side-steps the format mismatch. It
#      also guards `keras_tuner` / `GW_helper` imports and tweaks the
#      Wigner-d matrix to be JAX-traceable.
#
# Once `mlgw_bbh_jax` is properly committed to the GWjax repo (or pinned
# as a submodule), this whole cell can be deleted.
import base64, subprocess, sys, shutil, tempfile
from pathlib import Path

# 1. Install training-only deps that the upstream sub-repo eagerly imports.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "keras-tuner"],
    check=True,
)

# 2. Clone the vendored sub-repo into the gwjax install path.
import gwjax
_GW   = Path(gwjax.__file__).resolve().parent
_DEST = _GW / "mlgw_jax" / "mlgw_bbh_jax"
_ALREADY_PATCHED = _DEST / ".gwjax_inference_patch_applied"

if _ALREADY_PATCHED.exists():
    print(f"✓ mlgw_bbh_jax already present and patched at {_DEST}")
else:
    if _DEST.exists():
        shutil.rmtree(_DEST)
    print(f"Cloning mlgw_bbh_jax into {_DEST} (~1 GB, ~1–2 min on Colab)…")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", "new_training",
        "https://github.com/adrianomascioli/MLGW-JAX.git",
        str(_DEST),
    ], check=True)
    print(f"  ↳ cloned")

    # 3. Apply the inference-only patch (base64-encoded unified diff).
    _PATCH_B64 = (
        "ZGlmZiAtLWdpdCBhL21sZ3cvR1dfZ2VuZXJhdG9yLnB5IGIvbWxndy9HV19nZW5lcmF0b3IucHkKaW5kZXgg"
        "NGYxM2I0NC4uZDBiNGNjMiAxMDA2NDQKLS0tIGEvbWxndy9HV19nZW5lcmF0b3IucHkKKysrIGIvbWxndy9H"
        "V19nZW5lcmF0b3IucHkKQEAgLTI4LDcgKzI4LDEwIEBAIGltcG9ydCBudW1weSBhcyBucAogaW1wb3J0IGFz"
        "dAogaW1wb3J0IHRlbnNvcmZsb3cgYXMgdGYKIGZyb20gdGVuc29yZmxvdy5rZXJhcyBpbXBvcnQgbW9kZWxz"
        "IGFzIGtlcmFzX21vZGVscwotZnJvbSB0ZW5zb3JmbG93LnB5dGhvbi5mcmFtZXdvcmsuY29udmVydF90b19j"
        "b25zdGFudHMgaW1wb3J0IGNvbnZlcnRfdmFyaWFibGVzX3RvX2NvbnN0YW50c192MgordHJ5OiAgIyBpbnRl"
        "cm5hbCBBUEkgcGF0aCBtb3ZlZCBpbiBURiA+PSAyLjE2OyBndWFyZCBmb3IgZm9yd2FyZCBjb21wYXRpYmls"
        "aXR5CisgICAgZnJvbSB0ZW5zb3JmbG93LnB5dGhvbi5mcmFtZXdvcmsuY29udmVydF90b19jb25zdGFudHMg"
        "aW1wb3J0IGNvbnZlcnRfdmFyaWFibGVzX3RvX2NvbnN0YW50c192MgorZXhjZXB0IEltcG9ydEVycm9yOgor"
        "ICAgIGNvbnZlcnRfdmFyaWFibGVzX3RvX2NvbnN0YW50c192MiA9IE5vbmUKIGltcG9ydCBpbnNwZWN0CiBz"
        "eXMucGF0aC5pbnNlcnQoMSwgb3MucGF0aC5kaXJuYW1lKF9fZmlsZV9fKSkgCSNhZGRpbmcgdG8gcGF0aCBm"
        "b2xkZXIgd2hlcmUgbWxndyBwYWNrYWdlIGlzIGluc3RhbGxlZCAodWdseT8pCiBmcm9tIC5FTV9Nb0UgaW1w"
        "b3J0IE1vRV9tb2RlbCAjV0FSTklORyBjb21tZW50ZWQgb3V0IApAQCAtMTIzOCw5ICsxMjQxLDkgQEAgY2xh"
        "c3MgR1dfZ2VuZXJhdG9yOgogCQkJCQlzaW5faCAqKiAobiAtIG0gKyAyICogcykgLyBcCiAJCQkJCShmYWN0"
        "KGwgKyBtIC0gcykgKiBmYWN0KHMpICoKIAkJCQkJZmFjdChuIC0gbSArIHMpICogZmFjdChsIC0gbiAtIHMp"
        "KQotCQkJcmV0dXJuIGFjYyArIGpheC5sYXguY29uZChpbl9yYW5nZSwgbGFtYmRhOiBqbnAuc3F1ZWV6ZSh0"
        "ZXJtKSwgbGFtYmRhOiBqbnAuYXJyYXkoMC4wKSkKKwkJCXJldHVybiBhY2MgKyBqbnAud2hlcmUoaW5fcmFu"
        "Z2UsIHRlcm0sIGpucC56ZXJvc19saWtlKHRlcm0pKQogCi0JCXRvdGFsID0gamF4LmxheC5mb3JpX2xvb3Ao"
        "MCwgMiAqIGwgKyAxLCBib2R5LCAwLjApCisJCXRvdGFsID0gamF4LmxheC5mb3JpX2xvb3AoMCwgMiAqIGwg"
        "KyAxLCBib2R5LCBqbnAuemVyb3NfbGlrZShjb3NfaCkpCiAJCXJldHVybiBwcmVmICogdG90YWwKIAogCScn"
        "JwpkaWZmIC0tZ2l0IGEvbWxndy9OTl9tb2RlbC5weSBiL21sZ3cvTk5fbW9kZWwucHkKaW5kZXggYzJmMWU5"
        "MS4uNzZlYWQ3NiAxMDA2NDQKLS0tIGEvbWxndy9OTl9tb2RlbC5weQorKysgYi9tbGd3L05OX21vZGVsLnB5"
        "CkBAIC0xNCwxMCArMTQsMjMgQEAgaW1wb3J0IGdsb2IKIAogb3MuZW52aXJvblsnVEZfQ1BQX01JTl9MT0df"
        "TEVWRUwnXSA9ICcyJwogCi1mcm9tIGtlcmFzX3R1bmVyIGltcG9ydCBCYXllc2lhbk9wdGltaXphdGlvbiwg"
        "SHlwZXJNb2RlbAorIyBrZXJhc190dW5lciBpcyBvbmx5IG5lZWRlZCBmb3IgdHJhaW5pbmcg4oCUIG1ha2Ug"
        "aXQgb3B0aW9uYWwgZm9yIGluZmVyZW5jZQordHJ5OgorICAgIGZyb20ga2VyYXNfdHVuZXIgaW1wb3J0IEJh"
        "eWVzaWFuT3B0aW1pemF0aW9uLCBIeXBlck1vZGVsCisgICAgX2tlcmFzX3R1bmVyX2F2YWlsYWJsZSA9IFRy"
        "dWUKK2V4Y2VwdCBJbXBvcnRFcnJvcjoKKyAgICBfa2VyYXNfdHVuZXJfYXZhaWxhYmxlID0gRmFsc2UKKyAg"
        "ICBIeXBlck1vZGVsID0gb2JqZWN0ICAjIGZhbGxiYWNrIHNvIHRoZSBOTl9IeXBlck1vZGVsIGNsYXNzIGRl"
        "ZmluaXRpb24gZG9lc24ndCBjcmFzaAorCiBpbXBvcnQgdGVuc29yZmxvdyBhcyB0ZgogZnJvbSB0ZW5zb3Jm"
        "bG93IGltcG9ydCBrZXJhcwotZnJvbSBHV19oZWxwZXIgaW1wb3J0IGNvbXB1dGVfb3B0aW1hbF9taXNtYXRj"
        "aAorCisjIEdXX2hlbHBlci5jb21wdXRlX29wdGltYWxfbWlzbWF0Y2ggaXMgb25seSB1c2VkIGR1cmluZyB0"
        "cmFpbmluZwordHJ5OgorICAgIGZyb20gR1dfaGVscGVyIGltcG9ydCBjb21wdXRlX29wdGltYWxfbWlzbWF0"
        "Y2gKK2V4Y2VwdCBJbXBvcnRFcnJvcjoKKyAgICBjb21wdXRlX29wdGltYWxfbWlzbWF0Y2ggPSBOb25lCisK"
        "IGZyb20gTUxfcm91dGluZXMgaW1wb3J0IFBDQV9tb2RlbCwgYXVnbWVudF9mZWF0dXJlcwogZnJvbSBrZXJh"
        "cy5sYXllcnMgaW1wb3J0IERlbnNlCiBmcm9tIGtlcmFzLm9wdGltaXplcnMgaW1wb3J0IE5hZGFtCkBAIC0z"
        "NjksOSArMzgyLDczIEBAIGNsYXNzIG1sZ3dfTk4oa2VyYXMuU2VxdWVudGlhbCk6CiAJCQogCQlyZXR1cm4g"
        "bW9kZWwKIAorCUBjbGFzc21ldGhvZAorCWRlZiBfYnVpbGRfZnJvbV9sYXllcl9zcGVjcyhjbHMsIGNmZyk6"
        "CisJCSIiIgorCQlCdWlsZCBhIG1sZ3dfTk4gbW9kZWwgYnkgcGFyc2luZyB0aGUgbGF5ZXJzIGxpc3QgZGly"
        "ZWN0bHkuCisJCUhhbmRsZXMgYm90aCBLZXJhcyAyLnggKGJhdGNoX2lucHV0X3NoYXBlKSBhbmQgMy54IChi"
        "YXRjaF9zaGFwZS9EVHlwZVBvbGljeSkKKwkJY29uZmlnIGZvcm1hdHMgd2l0aG91dCByZWx5aW5nIG9uIGtl"
        "cmFzLlNlcXVlbnRpYWwuZnJvbV9jb25maWcoKS4KKwkJIiIiCisJCWZyb20ga2VyYXMubGF5ZXJzIGltcG9y"
        "dCBEZW5zZSBhcyBfRGVuc2UKKworCQlmZWF0dXJlcyA9IGNmZy5nZXQoJ2ZlYXR1cmVzJykgb3IgW10KKwkJ"
        "aWYgaXNpbnN0YW5jZShmZWF0dXJlcywgc3RyKToKKwkJCWZlYXR1cmVzID0gW2ZlYXR1cmVzXQorCisJCWlu"
        "cHV0X3NoYXBlID0gTm9uZQorCQlkZW5zZV9zcGVjcyA9IFtdCisKKwkJZm9yIGxheWVyX2NmZyBpbiBjZmcu"
        "Z2V0KCdsYXllcnMnLCBbXSk6CisJCQljbGFzc19uYW1lID0gbGF5ZXJfY2ZnLmdldCgnY2xhc3NfbmFtZScs"
        "ICcnKQorCQkJbGMgPSBsYXllcl9jZmcuZ2V0KCdjb25maWcnLCB7fSkKKworCQkJaWYgY2xhc3NfbmFtZSA9"
        "PSAnSW5wdXRMYXllcic6CisJCQkJYnMgPSBsYy5nZXQoJ2JhdGNoX3NoYXBlJykgb3IgbGMuZ2V0KCdiYXRj"
        "aF9pbnB1dF9zaGFwZScpCisJCQkJaWYgYnM6CisJCQkJCWlucHV0X3NoYXBlID0gdHVwbGUocyBmb3IgcyBp"
        "biBicyBpZiBzIGlzIG5vdCBOb25lKSAgIyBkcm9wIE5vbmUgYmF0Y2ggZGltCisJCQllbGlmIGNsYXNzX25h"
        "bWUgPT0gJ0RlbnNlJzoKKwkJCQl1bml0cyA9IGxjWyd1bml0cyddCisJCQkJYWN0ID0gbGMuZ2V0KCdhY3Rp"
        "dmF0aW9uJywgJ2xpbmVhcicpCisJCQkJIyBLZXJhcyAzLnggbWF5IHN0b3JlIGFjdGl2YXRpb24gYXMgYSBk"
        "aWN0OyBleHRyYWN0IHRoZSBuYW1lCisJCQkJaWYgaXNpbnN0YW5jZShhY3QsIGRpY3QpOgorCQkJCQlhY3Qg"
        "PSBhY3QuZ2V0KCdjb25maWcnLCB7fSkuZ2V0KCdhY3RpdmF0aW9uJykgb3IgYWN0LmdldCgnY2xhc3NfbmFt"
        "ZScsICdsaW5lYXInKQorCQkJCWRlbnNlX3NwZWNzLmFwcGVuZCgodW5pdHMsIGFjdCkpCisKKwkJbW9kZWwg"
        "PSBjbHMobmFtZT1jZmcuZ2V0KCduYW1lJywgJ21sZ3dfbm4nKSwgZmVhdHVyZXM9ZmVhdHVyZXMpCisJCWlm"
        "IGlucHV0X3NoYXBlOgorCQkJbW9kZWwuYWRkKGtlcmFzLmxheWVycy5JbnB1dExheWVyKGlucHV0X3NoYXBl"
        "PWlucHV0X3NoYXBlKSkKKwkJZm9yIHVuaXRzLCBhY3RpdmF0aW9uIGluIGRlbnNlX3NwZWNzOgorCQkJbW9k"
        "ZWwuYWRkKF9EZW5zZSh1bml0cywgYWN0aXZhdGlvbj1hY3RpdmF0aW9uKSkKKworCQlyZXR1cm4gbW9kZWwK"
        "KwogCUBjbGFzc21ldGhvZAogCWRlZiBsb2FkX2Zyb21fZmlsZShjbHMsIG5uX2ZpbGUpOgotCQlyZXR1cm4g"
        "a2VyYXMubW9kZWxzLmxvYWRfbW9kZWwobm5fZmlsZSwgY3VzdG9tX29iamVjdHM9eyJtbGd3X05OIiA6IGNs"
        "c30sIGNvbXBpbGU9RmFsc2UpCisJCWltcG9ydCB6aXBmaWxlIGFzIF96ZiwganNvbiBhcyBfanNvbiwgaW8g"
        "YXMgX2lvLCBoNXB5IGFzIF9oNXB5LCBudW1weSBhcyBfbnAKKworCQl3aXRoIF96Zi5aaXBGaWxlKG5uX2Zp"
        "bGUsICdyJykgYXMgejoKKwkJCWNvbmZpZyA9IF9qc29uLmxvYWRzKHoucmVhZCgnY29uZmlnLmpzb24nKSkK"
        "KwkJCXdlaWdodHNfYnl0ZXMgPSB6LnJlYWQoJ21vZGVsLndlaWdodHMuaDUnKQorCisJCW1vZGVsID0gY2xz"
        "Ll9idWlsZF9mcm9tX2xheWVyX3NwZWNzKGNvbmZpZ1snY29uZmlnJ10pCisKKwkJIyBXZWlnaHRzIGluIC5r"
        "ZXJhcyBmaWxlcyBhcmUgc3RvcmVkIHVuZGVyCisJCSMgX2xheWVyX2NoZWNrcG9pbnRfZGVwZW5kZW5jaWVz"
        "LzxuYW1lPi92YXJzLzxpPi4KKwkJIyBMYXllciBhdXRvLW5hbWVzIGF0IHNhdmUgYW5kIGxvYWQgdGltZSBt"
        "YXkgZGlmZmVyLCBzbyBhc3NpZ24gcG9zaXRpb25hbGx5LgorCQl3aXRoIF9oNXB5LkZpbGUoX2lvLkJ5dGVz"
        "SU8od2VpZ2h0c19ieXRlcyksICdyJykgYXMgaDoKKwkJCWRlcHMgPSBoLmdldCgnX2xheWVyX2NoZWNrcG9p"
        "bnRfZGVwZW5kZW5jaWVzJykKKwkJCWlmIGRlcHMgaXMgbm90IE5vbmU6CisJCQkJYWxsX3dlaWdodHMgPSBb"
        "XQorCQkJCWZvciBkZXBfa2V5IGluIHNvcnRlZChkZXBzLmtleXMoKSk6CisJCQkJCWlmICd2YXJzJyBpbiBk"
        "ZXBzW2RlcF9rZXldOgorCQkJCQkJdmFyc19ncnAgPSBkZXBzW2RlcF9rZXldWyd2YXJzJ10KKwkJCQkJCWZv"
        "ciBpIGluIHNvcnRlZCh2YXJzX2dycC5rZXlzKCksIGtleT1pbnQpOgorCQkJCQkJCWFsbF93ZWlnaHRzLmFw"
        "cGVuZChfbnAuYXJyYXkodmFyc19ncnBbaV0pKQorCQkJCW1vZGVsX3dlaWdodHMgPSBbdyBmb3IgbGF5ZXIg"
        "aW4gbW9kZWwubGF5ZXJzIGZvciB3IGluIGxheWVyLndlaWdodHNdCisJCQkJaWYgbGVuKGFsbF93ZWlnaHRz"
        "KSA9PSBsZW4obW9kZWxfd2VpZ2h0cyk6CisJCQkJCWZvciB3LCBkYXRhIGluIHppcChtb2RlbF93ZWlnaHRz"
        "LCBhbGxfd2VpZ2h0cyk6CisJCQkJCQl3LmFzc2lnbihkYXRhKQorCQlyZXR1cm4gbW9kZWwKIAogIwkjVGhp"
        "cyBpcyBicm9rZW4sIGxvYWRfd2VpZ2h0c19hbmRfZmVhdHVyZXMgZG9lc24ndCBleGlzdCBhbnltb3JlCiAj"
        "CUBjbGFzc21ldGhvZAo="
    )
    print("Applying inference-only patch (Keras 2/3 model-load compat + import guards)…")
    with tempfile.NamedTemporaryFile("wb", suffix=".patch", delete=False) as f:
        f.write(base64.b64decode(_PATCH_B64))
        _patch_path = f.name
    subprocess.run(
        ["git", "-C", str(_DEST), "apply", "--whitespace=nowarn", _patch_path],
        check=True,
    )
    _ALREADY_PATCHED.touch()
    print(f"✓ mlgw_bbh_jax patched at {_DEST}")

# 4. Verify the wrapper can now import the vendored Python package.
from gwjax.mlgw_jax.mlgw_bbh_jax_waveform_generator import MLGWBBHGenerator  # noqa: F401
print("✓ MLGWBBHGenerator importable")

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")  # silence TF startup spam

import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Enable double precision — important for likelihood accuracy.
jax.config.update("jax_enable_x64", True)

import gwjax
print("gwjax version :", gwjax.__version__)
print("JAX devices   :", jax.devices())

## 2. Build the detector network (GW150914 standard window)

GW150914 was observed by H1 and L1. We use the LVC-standard analysis
window for this event: **4 s @ 4 kHz** in the **20–1024 Hz** band. The
`TimeFrequencyGrid` is shared across all detectors in the network.

In [ ]:
DURATION       = 4.0
SAMPLING_RATE  = 4096.0
F_MIN, F_MAX   = 20.0, 1024.0

grid = gwjax.TimeFrequencyGrid(
    duration=DURATION, sampling_rate=SAMPLING_RATE,
    f_min=F_MIN, f_max=F_MAX,
)
network = gwjax.Network.from_names(["H1", "L1"], grid)
print(grid)
print(network)

## 3. Fetch real GW150914 strain from GWOSC

`gwjax.compat.attach_event_to_network` does three things in one shot:

1. Downloads H1 and L1 strain around the GW150914 trigger via
   `gwpy.TimeSeries.fetch_open_data` (needs an internet connection).
2. Estimates each IFO's PSD from an **adjacent off-source** segment (32 s
   ending 8 s before the trigger) with Welch's method.
3. Crops + resamples the on-source segment so the **merger sits at
   0.875 × duration = 3.5 s** into the segment, then attaches strain
   and PSD to every IFO in the network.

The 3.5 s offset is the convention the GWjax `tc` parameter must match
(see cell 11).

In [ ]:
import time

t0 = time.perf_counter()
gwjax.compat.attach_event_to_network(
    network, "GW150914",
    estimate_psd          = True,
    psd_segment_duration  = 32.0,
    psd_offset            = 8.0,
    verbose               = False,
)
print(f"GW150914 strain + PSDs attached in {time.perf_counter()-t0:.1f} s")
print(f"  populated detectors : {[ifo.name for ifo in network.interferometers]}")

## 4. Build a `(params, freqs) -> (h₊, hₓ)` waveform_fn from mlgw_bbh_jax

`MLGWBBHGenerator` is a **time-domain** waveform model: it returns h₊(t),
hₓ(t) on a user-supplied time grid with the merger placed at t = 0
(the **end** of the segment in mlgw's convention). The GWjax sampler
however expects an **FD** waveform `(params, freqs) -> (ĥ₊, ĥₓ)`. The
shim below does exactly that: it evaluates the surrogate on
`grid.time_domain_array_mlgw` (length `N = duration × sampling_rate`,
spanning `[-duration, -dt]`) and rFFTs the result with the GWjax
convention `ĥ(f_k) = rfft(h)[k] × dt`.

Two notes about the parameter mapping:

* `mlgw_bbh_jax` requires **m₁ ≥ m₂**. We enforce this *inside* the
  waveform_fn by swapping `(m₁, m₂)` and `(χ₁, χ₂)` together with
  `jnp.where`, so the prior box is just a hyper-rectangle and NS can
  sample freely on both sides of `m₁ = m₂` without ever evaluating the
  surrogate outside its support. The physical interpretation of the
  posterior samples is then `(m_heavy, m_light, χ_heavy, χ_light)`.
* `phi_c` in our particle dict feeds the surrogate's reference orbital
  phase `phi_0` and the projection-layer phase — we just pass it
  through (and fix it to 0 by default).

The model loads (TensorFlow → tf2jax) on first construction; the
all-modes JIT path is cached internally so repeated calls with the same
time grid don't trigger recompilation.

In [ ]:
from gwjax.mlgw_jax.mlgw_bbh_jax_waveform_generator import MLGWBBHGenerator

def build_mlgw_bbh_waveform_fn(grid, modes=None):
    """Return a JAX-traceable ``(params, freqs) -> (hp, hc)`` callable.

    The TD surrogate is evaluated on ``grid.time_domain_array_mlgw`` and
    rFFT'd with the GWjax convention ĥ(f_k) = rfft(h)[k] · dt.  Particles
    with m1 < m2 are automatically swapped (with their spins) so the
    surrogate is always evaluated in its q ∈ [1, 10] domain.

    NOTE on the ``model=4`` argument
    --------------------------------
    The wrapper's default is ``model="model_4"`` (a string), but the
    underlying ``mlgw.GW_generator`` only resolves the bundled
    ``<mlgw>/TD_models/model_<N>`` path when an **int** is passed; strings
    are interpreted as literal absolute/relative paths and fail with
    ``Unable to load folder model_4: no such directory!``. Passing
    ``model=4`` triggers the int branch and finds the SEOBNRv5HM weights
    correctly. The canonical wrapper would benefit from defaulting to
    ``model=4`` itself; this notebook overrides it explicitly.
    """
    bbh    = MLGWBBHGenerator(model=4)
    t_grid = grid.time_domain_array_mlgw
    dt     = grid.dt
    gen_wf = bbh.generate  # = self._gen.get_WF (pure JAX, traceable)

    def waveform_fn(params, freqs):
        # Enforce m1 >= m2 (mlgw q = m1/m2 in [1, 10]) by swapping with spins.
        m1_in, m2_in       = params["m1"],    params["m2"]
        chi1_in, chi2_in   = params["chi_1"], params["chi_2"]
        swap   = m1_in < m2_in
        m1     = jnp.where(swap, m2_in,   m1_in)
        m2     = jnp.where(swap, m1_in,   m2_in)
        chi_1  = jnp.where(swap, chi2_in, chi1_in)
        chi_2  = jnp.where(swap, chi1_in, chi2_in)

        # theta = [m1, m2, s1z, s2z, d_L, inclination, phi_0]
        theta = jnp.stack([
            m1, m2,
            chi_1, chi_2,
            params["distance"],
            params["inclination"],
            params["phi_c"],
        ])
        hp_td, hc_td = gen_wf(theta, t_grid, modes=modes)
        hp_fd = jnp.fft.rfft(hp_td) * dt
        hc_fd = jnp.fft.rfft(hc_td) * dt
        return hp_fd, hc_fd

    return waveform_fn

t0 = time.perf_counter()
waveform_fn = build_mlgw_bbh_waveform_fn(grid)
print(f"mlgw_bbh_jax (SEOBNRv5HM, model_4) loaded in {time.perf_counter()-t0:.1f} s")

## 5. Set up the two-phase nested sampler

10-dimensional uniform prior. Differences vs. the ripplegw two-phase
notebook:

* **Spins are sampled**, not fixed: `chi_1, chi_2 ∈ [-0.9, 0.9]` (the
  full validity range of `mlgw_bbh_jax`).
* The `tc` prior is centred at **3.5 s = 0.875 × duration**, matching
  where `attach_event_to_network` places the merger in the cropped
  segment (a half-second window comfortably covers the trigger
  uncertainty).
* The mass prior is the usual ripplegw range; the waveform_fn enforces
  m₁ ≥ m₂ internally.
* `phi_c` is fixed to 0 (its only role is the constant phase offset on
  the surrogate, which is degenerate with `psi` at the dominant 2,2
  mode).

Published-median values from GWTC-1 are kept as plot overlays only; they
are *not* used as priors or as a truth target.

In [ ]:
TC_CENTER = 0.875 * grid.duration   # 3.5 s for the 4-s GW150914 segment

PARAM_BOUNDS = {
    "m1":          (10.0, 80.0),
    "m2":          (10.0, 80.0),
    "chi_1":       (-0.9, 0.9),
    "chi_2":       (-0.9, 0.9),
    "distance":    (50.0, 2000.0),
    "inclination": (0.0, float(jnp.pi)),
    "ra":          (0.0, 2.0 * float(jnp.pi)),
    "dec":         (-float(jnp.pi) / 2, float(jnp.pi) / 2),
    "psi":         (0.0, float(jnp.pi)),
    "tc":          (TC_CENTER - 0.5, TC_CENTER + 0.5),
}
FIXED_PARAMS = {"phi_c": 0.0}

# Published-median reference (GWTC-1) for the corner-plot overlay only.
PUBLISHED_MEDIAN = dict(
    m1=35.6, m2=30.6,
    chi_1=0.0, chi_2=0.0,
    distance=440.0, inclination=2.5,
    ra=2.21, dec=-1.25, psi=0.0,
    tc=TC_CENTER,
)

sampler = gwjax.GWjaxTwoPhaseNestedSampler(
    network       = network,
    waveform_fn   = waveform_fn,
    param_bounds  = PARAM_BOUNDS,
    fixed_params  = FIXED_PARAMS,
    gmst          = 0.0,
)
print(f"sampling dimension : {len(sampler.param_bounds)}")
print(f"free parameters    : {list(sampler.param_bounds)}")
print(f"fixed parameters   : {FIXED_PARAMS}")

## 6. Run the two-phase sampler

Same tuning knobs as the ripplegw two-phase notebook:

- `phase1_num_delete` between `num_live // 100` (very safe) and
  `num_live // 20` (faster, slight bias). We use `num_live // 20 = 25`.
- `phase1_delta_logz_threshold = -1.0` switches to phase 2 when the
  live points still hold ≈ 37 % of total `Z`.
- `phase2_num_delete = 1` is the classical Skilling choice (unbiased).
- `log_dlogz_target = -3.0` is phase-2's final convergence tolerance.
- `num_inner_steps = 50` (≈ 5·d for d = 10).

**Compilation warning.** Each phase compiles a separate XLA program
(`num_delete` is baked into the program); on top of that, the
SEOBNRv5HM tf2jax kernel itself is non-trivial, so the first compile in
particular can take **2–3 minutes** even on a T4. Steady-state
iterations are still fast (the model is a small MLP); plan for a total
wall-clock of ≈10–20 minutes at the settings below.

In [ ]:
NUM_LIVE        = 500
NUM_INNER_STEPS = 50    # ~5*d for d=10

t0 = time.perf_counter()
result = sampler.run_two_phase(
    rng_key                      = jax.random.PRNGKey(0),
    num_live                     = NUM_LIVE,
    num_inner_steps              = NUM_INNER_STEPS,

    # Phase 1 (bulk, batched delete)
    phase1_num_delete            = max(1, NUM_LIVE // 20),   # = 25 here
    phase1_delta_logz_threshold  = -1.0,
    phase1_max_iterations        = 1500,

    # Phase 2 (Skilling tail, classical)
    phase2_num_delete            = 1,
    phase2_max_iterations        = 8000,
    log_dlogz_target             = -3.0,

    num_posterior_samples        = 2000,
    verbose                      = True,
)
elapsed = time.perf_counter() - t0

print(f"\ntotal wall-clock         = {elapsed:.1f} s")
print(f"  phase-1 iterations     = {result.phase1_iterations}")
print(f"  phase-2 iterations     = {result.phase2_iterations}")
print(f"  log Z                  = {result.logZ:+.3f} ± {result.logZ_err:.3f}")
print(f"  ESS                    = {result.ess:.1f}")

## 7. Posterior summary

The "reference" column is the GWTC-1 published median (visual aid only,
not a truth target). If you see ⚠ collapsed columns / very low ESS
below, the NS budget was too small for the 10-D problem — bump
`NUM_LIVE` to 800–1000 or `NUM_INNER_STEPS` to 60 and re-run cells 11
→ 13.

In [ ]:
print(f"  {'param':12s}  {'median':>10s}  {'-1σ':>8s}  {'+1σ':>8s}  reference")
degenerate = []
for name in sampler.param_bounds:
    s = np.asarray(result.posterior_samples[name])
    lo, mid, hi = np.percentile(s, [16, 50, 84])
    if (hi - lo) < 1e-12 * max(abs(mid), 1.0):
        degenerate.append(name)
    ref_str = f"{PUBLISHED_MEDIAN[name]:+.3f}"
    print(f"  {name:12s}  {mid:+10.3f}  {mid-lo:8.3f}  {hi-mid:8.3f}  {ref_str}")

if degenerate:
    print(
        f"\n⚠  posterior columns collapsed (no spread): {degenerate}\n"
        f"   ESS={result.ess:.1f}  →  NS likely didn't converge.\n"
        f"   Bump NUM_LIVE to 800 and NUM_INNER_STEPS to 60, then re-run cell 13."
    )

## 8. Corner plot

The plot range is forced to the prior bounds via `range=`, so the plot
always renders even when the posterior is degenerate. Red lines are the
GWTC-1 published medians — visual reference, **not** truth.

Note on `m1, m2`: the waveform_fn enforces m₁ ≥ m₂, so the (m₁, m₂)
panel will show samples populating *only* the m₁ ≥ m₂ half-plane
(modulo the small fraction that NS draws in the swapped half before the
swap is applied) — this is the physically correct behaviour.

In [ ]:
import corner

names  = list(sampler.param_bounds.keys())
data   = np.column_stack([np.asarray(result.posterior_samples[n]) for n in names])
truths = [PUBLISHED_MEDIAN[n] for n in names]
ranges = [sampler.param_bounds[n] for n in names]

fig = corner.corner(
    data, labels=names, truths=truths,
    range=ranges,
    quantiles=[0.16, 0.5, 0.84], show_titles=True,
    title_kwargs={"fontsize": 10},
    truth_color="C3",
)
fig.set_size_inches(13, 13)
plt.show()

## What next?

- **Higher-fidelity budget**: bump `NUM_LIVE` to 1000 and
  `NUM_INNER_STEPS` to 60. Doubles the wall-clock but cleans up the
  spin and inclination posteriors substantially.
- **Restrict to the leading mode**: pass `modes=(2, 2)` to
  `build_mlgw_bbh_waveform_fn` to get the dominant-mode SEOBNRv5HM
  surrogate — ≈3× faster than the full 7-mode evaluation, with
  essentially no bias for an event like GW150914 that is consistent
  with non-precessing q ≈ 1.
- **Tighter prior**: shrink `m1, m2` to (20, 60) and the `tc` window to
  ±0.05 s once you've localised the merger — NS converges much faster.
- **Other events**: any registered event in
  `gwjax.compat.KNOWN_EVENTS` works — just change the event name in
  cell 7. Update `EVENT_SETTINGS` (see
  [`examples/run_pe_local_realdata.py`](https://github.com/saulo-albuquerque-phys/GWjax/blob/main/examples/run_pe_local_realdata.py))
  if the event needs a different `duration`, `sampling_rate`, or band
  (GW190521 wants 8 s starting at 11 Hz; GW170817 is BNS and needs
  `mlgw_bns_jax` rather than `mlgw_bbh_jax`).
- **Compare with ripplegw**: the canonical two-phase notebook is
  [`examples/gwjax_colab_pe_two_phase.ipynb`](https://github.com/saulo-albuquerque-phys/GWjax/blob/main/examples/gwjax_colab_pe_two_phase.ipynb)
  — same sampler, IMRPhenomD instead of SEOBNRv5HM, fixed spins. The
  GW150914 posterior should agree to within statistical error in mass
  and distance; the spin posteriors here add genuinely new information.